# Data Lineage (from utl_pipeline_metadata)
Zero API calls — reads `source_tables` column populated by engines.

In [ ]:
# ── Table Lineage ──
df = spark.sql("""
    SELECT table_name, layer, source_tables, load_type, status, last_load_date
    FROM SupplyChain_Lakehouse.dbo.utl_pipeline_metadata
    WHERE source_tables IS NOT NULL
    ORDER BY layer, table_name
""")
print(f"Tables with lineage: {df.count()}")
df.show(100, truncate=False)

In [5]:
# ── Cross-Table Links (producer → consumer) ──
df_links = spark.sql("""
    SELECT
        expanded.source_tbl   AS source_table,
        p.layer               AS source_layer,
        expanded.table_name   AS target_table,
        expanded.layer        AS target_layer,
        expanded.load_type
    FROM (
        SELECT c.table_name, c.layer, c.load_type, src.source_tbl
        FROM SupplyChain_Lakehouse.dbo.utl_pipeline_metadata c
        LATERAL VIEW EXPLODE(SPLIT(c.source_tables, ',')) src AS source_tbl
        WHERE c.source_tables IS NOT NULL
          AND src.source_tbl NOT LIKE '[external]%'
    ) expanded
    LEFT JOIN SupplyChain_Lakehouse.dbo.utl_pipeline_metadata p
        ON p.table_name = expanded.source_tbl
    ORDER BY source_table, target_table
""")
print(f"Cross-table links: {df_links.count()}")
df_links.show(100, truncate=False)

StatementMeta(, 81e9c2b7-ee8b-4698-be73-b7057d611a8c, 7, Finished, Available, Finished, False)

Cross-table links: 33
+-------------------------------------------------------+------------+-----------------------------+------------+---------+
|source_table                                           |source_layer|target_table                 |target_layer|load_type|
+-------------------------------------------------------+------------+-----------------------------+------------+---------+
|brz_saleshistory_afi__invoicedetail_ver2               |NULL        |slv_invoice_detail_line_level|SLV         |overwrite|
|brz_saleshistory_afi__invoiceheader_ver2               |NULL        |slv_invoice_detail_line_level|SLV         |overwrite|
|brz_supplychain_enh_1__demandforecastsnapshotdaily_ver2|NULL        |slv_forecast_demand_monthly  |SLV         |overwrite|
|brz_wholesale_codis_afi__codatan                       |BRZ         |slv_open_order_line_level    |SLV         |overwrite|
|brz_wholesale_codis_afi__comast                        |BRZ         |slv_open_order_line_level    |SLV       

In [6]:
# ── Lineage Summary by Layer ──
spark.sql("""
    SELECT
        layer,
        COUNT(*) AS notebook_count,
        SUM(CASE WHEN source_tables IS NOT NULL THEN 1 ELSE 0 END) AS with_lineage,
        SUM(CASE WHEN status = 'success' THEN 1 ELSE 0 END) AS healthy
    FROM SupplyChain_Lakehouse.dbo.utl_pipeline_metadata
    GROUP BY layer
    ORDER BY layer
""").show()

StatementMeta(, 81e9c2b7-ee8b-4698-be73-b7057d611a8c, 8, Finished, Available, Finished, False)

+-----+--------------+------------+-------+
|layer|notebook_count|with_lineage|healthy|
+-----+--------------+------------+-------+
|  BRZ|             7|           7|      7|
|  GLD|             2|           2|      2|
|  REF|             9|           9|      9|
|  SLV|             8|           8|      8|
+-----+--------------+------------+-------+



In [7]:
# ── Orphan Tables (source referenced but no producer registered) ──
spark.sql("""
    SELECT DISTINCT expanded.source_tbl AS orphan_table
    FROM (
        SELECT src.source_tbl
        FROM SupplyChain_Lakehouse.dbo.utl_pipeline_metadata c
        LATERAL VIEW EXPLODE(SPLIT(c.source_tables, ',')) src AS source_tbl
        WHERE c.source_tables IS NOT NULL
          AND src.source_tbl NOT LIKE '[external]%'
    ) expanded
    LEFT JOIN SupplyChain_Lakehouse.dbo.utl_pipeline_metadata p
        ON p.table_name = expanded.source_tbl
    WHERE p.table_name IS NULL
    ORDER BY orphan_table
""").show(50, truncate=False)

StatementMeta(, 81e9c2b7-ee8b-4698-be73-b7057d611a8c, 9, Finished, Available, Finished, False)

+-------------------------------------------------------+
|orphan_table                                           |
+-------------------------------------------------------+
|brz_saleshistory_afi__invoicedetail_ver2               |
|brz_saleshistory_afi__invoiceheader_ver2               |
|brz_supplychain_enh_1__demandforecastsnapshotdaily_ver2|
|ref_forecast_cycle                                     |
+-------------------------------------------------------+

